In [ ]:
import pandas as pd

deposit_name_df = pd.read_excel("./data_raw/DEPOSIT_NAME.xlsx")
geol_structure_df = pd.read_excel("./data_raw/GEOL_STRUCTURE.xlsx")
mineral_deposit_df = pd.read_excel("./data_raw/MINERAL_DEPOSIT.xlsx")
township_area_df = pd.read_excel("./data_raw/TOWNSHIP_AREA.xlsx")
township_df = pd.read_excel("./data_raw/TOWNSHIP.xlsx")
zone_df = pd.read_excel("./data_raw/ZONE.xlsx")

acoustic_props_df = pd.read_csv("./data_raw/geophysical_rock_properties/acoustic_properties.csv")
densities_magetic_props_df = pd.read_csv("./data_raw/geophysical_rock_properties/densities_and_magnetic.csv")

## Merge mineral deposit datasets:

In [ ]:
deposit_name_df.drop(columns=["CREATED_AT", "UPDATED_AT"], inplace=True, errors="ignore")
deposit_name_df.columns

Index(['MD_ID', 'MD_NAME', 'RANKING', 'YEAR'], dtype='object')

In [41]:
geol_structure_df.drop(columns=["CREATED_AT", "UPDATED_AT"], inplace=True, errors="ignore")
geol_structure_df.columns

Index(['MD_ID', 'STRUCTURE_NAME', 'REGIONAL_LOCAL_FLG', 'STRUCTURE_STRIKE',
       'STRUCTURE_DIP', 'STRUCTURE_TREND', 'STRUCTURE_PLUNGE'],
      dtype='object')

In [42]:
mineral_deposit_df.drop(columns=["CREATED_AT", "UPDATED_AT"], inplace=True, errors="ignore")
mineral_deposit_df.columns

Index(['ID', 'MDI_IDENT', 'LOCATION_METHOD', 'POINT_LOCATION_DESCR',
       'UTM_DATUM', 'UTM_ZONE', 'UTM_EASTING', 'UTM_NORTHING', 'CREATE_DATE',
       'CREATE_GEOLOGIST_ID', 'REVISION_GEOLOGIST_ID', 'OLD_MDI_ID', 'SMDR_ID',
       'AMIS_ID', 'RELATED_DEPOSIT_FLAG', 'DEPOSIT_ACCESS_DESCR', 'SOURCE_MAP',
       'SOURCE_MAP_SCALE', 'SOURCE_MAP_ACCURACY', 'INTRUSION_NAME',
       'TERRANE_NAME', 'TECTONIC_ASSEMBLAGE_NAME', 'GEOLOGICAL_AGE_ID',
       'GEOCHRONOLOGICAL_AGE', 'GEOCHRONOLOGICAL_AGE_REF', 'METAMORPHISM_TYPE',
       'METAMORPHISM_GRADE', 'EXPLORATION_HISTORY', 'DEPOSIT_STATUS_ID',
       'LNG_DEC', 'LAT_DEC', 'LL_DATUM', 'PROVINCE_ID', 'SUBPROVINCE_ID',
       'BELT_ID', 'RGP_DISTRICT_ID', 'TERRANE_ID', 'SUPERIOR_SUBPROVINCE_ID',
       'DOMAIN_ID', 'ASSEMBLAGE_ID', 'SUPERGROUP_ID', 'FORMATION_GROUP_ID',
       'SOURCE_MAP_SCALE_ID'],
      dtype='object')

In [53]:
# merge mineral deposit data from 3 tables

mineral_deposits_df = pd.merge(mineral_deposit_df, deposit_name_df, left_on="ID", right_on="MD_ID", how="left")
mineral_deposits_df = mineral_deposits_df.drop(columns=["MD_ID"])

mineral_deposits_df = pd.merge(mineral_deposit_df, geol_structure_df, left_on="ID", right_on="MD_ID", how="left")
mineral_deposits_df = mineral_deposits_df.drop(columns=["MD_ID"])

mineral_deposits_df.columns

Index(['ID', 'MDI_IDENT', 'LOCATION_METHOD', 'POINT_LOCATION_DESCR',
       'UTM_DATUM', 'UTM_ZONE', 'UTM_EASTING', 'UTM_NORTHING', 'CREATE_DATE',
       'CREATE_GEOLOGIST_ID', 'REVISION_GEOLOGIST_ID', 'OLD_MDI_ID', 'SMDR_ID',
       'AMIS_ID', 'RELATED_DEPOSIT_FLAG', 'DEPOSIT_ACCESS_DESCR', 'SOURCE_MAP',
       'SOURCE_MAP_SCALE', 'SOURCE_MAP_ACCURACY', 'INTRUSION_NAME',
       'TERRANE_NAME', 'TECTONIC_ASSEMBLAGE_NAME', 'GEOLOGICAL_AGE_ID',
       'GEOCHRONOLOGICAL_AGE', 'GEOCHRONOLOGICAL_AGE_REF', 'METAMORPHISM_TYPE',
       'METAMORPHISM_GRADE', 'EXPLORATION_HISTORY', 'DEPOSIT_STATUS_ID',
       'LNG_DEC', 'LAT_DEC', 'LL_DATUM', 'PROVINCE_ID', 'SUBPROVINCE_ID',
       'BELT_ID', 'RGP_DISTRICT_ID', 'TERRANE_ID', 'SUPERIOR_SUBPROVINCE_ID',
       'DOMAIN_ID', 'ASSEMBLAGE_ID', 'SUPERGROUP_ID', 'FORMATION_GROUP_ID',
       'SOURCE_MAP_SCALE_ID', 'STRUCTURE_NAME', 'REGIONAL_LOCAL_FLG',
       'STRUCTURE_STRIKE', 'STRUCTURE_DIP', 'STRUCTURE_TREND',
       'STRUCTURE_PLUNGE'],
      dtype

In [ ]:
mineral_deposits_df.to_csv("./data_combined/ON_mineral_deposits.csv", index=False)

## Merge previous datasets with rock properties dataset (using nearest data point):

In [51]:
rock_properties_df = pd.read_csv("./data_raw/geophysical_rock_properties/densities_and_magnetic.csv")
rock_properties_df.columns

Index(['X', 'Y', 'SAMPLE', 'SOURCE', 'DEPTH', 'ELEVATION', 'ROCKTYPE',
       'COMMENTS', 'DENSITY', 'SMSNO', 'MAGSUS', 'LATITUDE', 'LONGITUDE',
       'SDMAGSUS'],
      dtype='object')

In [62]:
import geopandas as gpd

gdf1 = gpd.GeoDataFrame(
    mineral_deposits_df,
    geometry=gpd.points_from_xy(mineral_deposits_df["LNG_DEC"], mineral_deposits_df["LAT_DEC"]),
    crs="EPSG:4326"
)

gdf2 = gpd.GeoDataFrame(
    rock_properties_df,
    geometry=gpd.points_from_xy(rock_properties_df["LONGITUDE"], rock_properties_df["LATITUDE"]),
    crs="EPSG:4326"
)

result = gdf1.sjoin_nearest(gdf2, distance_col="distance")
result.drop(columns=['LATITUDE', 'LONGITUDE', 'X', 'Y'], inplace=True, errors="ignore")
result.shape

c:\Users\jylis\AppData\Local\Programs\Python\Python310\lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


(41626, 62)

In [65]:
result.to_csv("./data_combined/ON_combined_data.csv", index=False)